In [12]:
import os
import yaml
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

from utils import collect_dates, fetch_ids, parse_vacancies

%load_ext autoreload
%autoreload 2

# 1) Сбор ссылок на релевантные вакансии

In [2]:
load_dotenv()
APP_TOKEN = os.getenv("APP_TOKEN")
APP_NAME = os.getenv("APP_NAME")
EMAIL = os.getenv("EMAIL")
headers = {
    'User-Agent': f'{APP_NAME} ({EMAIL})', 
    "Authorization": f"Bearer {APP_TOKEN}"
    }

In [3]:
HH_PARSING_CONFIG_PATH = Path.cwd().parent / 'configs' / 'hh_parsing.yaml'
with HH_PARSING_CONFIG_PATH.open(encoding='utf-8') as f:
    parsing_config = yaml.safe_load(f)

vacancies_url = parsing_config.get('vacancies_url', '')
relevant_proles = parsing_config.get('relevant_proles', [])
# year = int(parsing_config.get('year', ''))
# month = int(parsing_config.get('month', ''))
n_pages_to_parse = int(parsing_config.get('n_pages_to_parse', ''))
n_vacancies_per_page = int(parsing_config.get('n_vacancies_per_page', ''))

### Собираем id вакансий, опубликованных с 21 по 31 августа 2026 года

In [ ]:
dates_august = collect_dates(2026, 8)
ids_august = fetch_ids(dates_august, relevant_proles, n_pages_to_parse, n_vacancies_per_page, vacancies_url, headers)
with open("ids_august.txt", "w") as f:
    for id in ids_august:
        f.write(id + "\n")

### Собираем id вакансий, опубликованных с 1 по 20 сентября 2026 года

In [ ]:
dates_september = collect_dates(2026, 9)
ids_september = fetch_ids(dates_september, relevant_proles, n_pages_to_parse, n_vacancies_per_page, vacancies_url, headers)
with open("ids_september.txt", "w") as f:
    for id in ids_september:
        f.write(id + "\n")

# 2) Парсинг страниц вакансий

In [23]:
ids_august = set()
with open("data/ids_august.txt", "r") as file:
    for line in file.readlines():
        ids_august.add(line.strip())

ids_september = set()
with open("data/ids_september.txt", "r") as file:
    for line in file.readlines():
        ids_september.add(line.strip())

ids_all = ids_august.union(ids_september)
with open("data/ids_all.txt", "w") as file:
    for id in ids_all:
        file.write(id + '\n')

In [ ]:
vacancies_df = parse_vacancies("data/ids_all.txt")

# 3) Удаление дубликатов

In [13]:
vacancies_raw = pd.read_csv("data/vacancies_raw.csv")

In [15]:
vacancies_raw.head()

,id,name,area,salary,salary_range,experience,description,key_skills,professional_role,employer,work_schedule
0,136714662,Графический дизайнер,Томск,"{'from': 45000, 'to': None, 'currency': 'RUR',...","{'from': 45000, 'to': None, 'currency': 'RUR',...",От 1 года до 3 лет,<p>Требуемый опыт работы: 1–3 года</p> <p>Полн...,[],34,Дельтаплан,5/2
1,136686210,Fullstack-разработчик,Владивосток,NaN,NaN,От 1 года до 3 лет,<h2><strong>Об Xsolla</strong></h2> <div><stro...,"[{'name': 'JavaScript'}, {'name': 'TypeScript'...",96,Xsolla,5/2
2,134824641,Аналитик информационной безопасности (г. Дзерж...,Дзержинск (Нижегородская область),NaN,NaN,От 3 до 6 лет,<p><strong>ГК «Синтез ОКА»</strong> является л...,"[{'name': 'Событийная аналитика'}, {'name': 'С...",116,Синтез ОКА,5/2
3,136725052,Разработчик C++,Новосибирск,NaN,NaN,От 1 года до 3 лет,<div><strong>ARQA Technologies —</strong> неза...,"[{'name': 'C++'}, {'name': 'MS Visual Studio'}...",96,ARQA Technologies,5/2
4,136795453,Ведущий инженер электросвязи (группа голосовой...,Минск,NaN,NaN,От 1 года до 3 лет,<p><strong>Основные задачи:</strong></p> <ul> ...,"[{'name': 'Голосовое ядро сети'}, {'name': 'Яд...",112,"Мобильные ТелеСистемы (МТС), Беларусь",5/2


In [16]:
vacancies_raw.shape

(40952, 11)

Посмотрим, сколько есть вакансий, дублирующих уже существующие по описанию

In [17]:
vacancies_raw[vacancies_raw.duplicated(subset = ["description"])].shape

(4578, 11)

Удалим дубликаты

In [24]:
vacancies_cleaned = vacancies_raw[~vacancies_raw.duplicated(subset = ["description"])]
vacancies_cleaned.to_csv("data/vacancies_cleaned.csv", index=False)